# DDSP-SVC ReFlow для Google Colab

Этот блокнот ставит актуальный DDSP-SVC, загружает обязательные предобученные модели и запускает русский Gradio интерфейс.

Перед запуском выбери GPU runtime. Для Tesla T4 стартовый batch size в интерфейсе установлен 32.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive подключён')


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1 unzip
!rm -rf /content/DDSP-SVC /content/DDSP-SVC-ReFlow-Colab
!mkdir -p /content/DDSP-SVC
!git -C /content/DDSP-SVC init -q
!git -C /content/DDSP-SVC remote add origin https://github.com/yxlllc/DDSP-SVC.git
!git -C /content/DDSP-SVC fetch -q --depth 1 origin 3635301027473c6662d05a1c73ef34fba7f15f90
!git -C /content/DDSP-SVC checkout -q --detach FETCH_HEAD
!git clone --depth 1 https://github.com/egor125552/DDSP-SVC-ReFlow-Colab.git /content/DDSP-SVC-ReFlow-Colab
%cd /content/DDSP-SVC
!git rev-parse HEAD
!python -m pip install -q --upgrade pip
!python /content/DDSP-SVC-ReFlow-Colab/scripts/make_colab_requirements.py requirements.txt /tmp/ddsp-requirements-colab.txt
!python -m pip install -q -r /tmp/ddsp-requirements-colab.txt
!python -m pip install -q -r /content/DDSP-SVC-ReFlow-Colab/requirements-extra.txt
!python - <<'PY'
import numpy, numba, resampy, torchcrepe
from ddsp.vocoder import F0_Extractor
print('Dependency smoke: OK')
print('NumPy:', numpy.__version__)
print('Numba:', numba.__version__)
print('resampy:', getattr(resampy, '__version__', 'unknown'))
print('torchcrepe import: OK')
print('ddsp.vocoder import: OK')
PY
print('Зависимости установлены и проверены')


In [ ]:
import os, shutil
from pathlib import Path
import torch

print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM, ГБ:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

drive_root = Path('/content/drive/MyDrive/DDSP-SVC-ReFlow')
drive_root.mkdir(parents=True, exist_ok=True)

for name in ('exp', 'data'):
    target = drive_root / name
    target.mkdir(exist_ok=True)
    link = Path('/content/DDSP-SVC') / name
    if link.exists() or link.is_symlink():
        if link.is_symlink() or link.is_file():
            link.unlink()
        else:
            shutil.rmtree(link)
    os.symlink(target, link, target_is_directory=True)

print('Чекпойнты будут сохраняться в', drive_root / 'exp')
print('Датасет и подготовленные признаки будут сохраняться в', drive_root / 'data')


In [ ]:
!bash /content/DDSP-SVC-ReFlow-Colab/scripts/download_pretrained.sh /content/DDSP-SVC
print('Предобученные модели готовы')


In [ ]:
import os
from pathlib import Path
import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA недоступна. В Colab выбери Среда выполнения → Изменить среду выполнения → GPU.')

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print('GPU smoke test')
print('GPU:', gpu)
print('VRAM, ГБ:', round(vram, 1))
print('Python:', '.'.join(map(str, __import__('sys').version_info[:3])))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)

tested = {
    'python_major_minor': (3, 12),
    'supported_python': {(3, 12), (3, 13)},
    'torch_prefix': '2.11.',
    'cuda_prefix': '12.8',
}
import sys
runtime_warnings = []
if sys.version_info[:2] not in tested['supported_python']:
    runtime_warnings.append(
        f"Python {sys.version_info.major}.{sys.version_info.minor}, поддерживаются 3.12 и 3.13"
    )
elif sys.version_info[:2] == (3, 13):
    print('Python 3.13: включён compatibility path для NumPy/Numba.')
if not str(torch.__version__).startswith(tested['torch_prefix']):
    runtime_warnings.append(f"PyTorch {torch.__version__}, тестировался PyTorch 2.11.x")
if not str(torch.version.cuda).startswith(tested['cuda_prefix']):
    runtime_warnings.append(f"CUDA {torch.version.cuda}, тестировалась CUDA 12.8")
if runtime_warnings:
    print('ПРЕДУПРЕЖДЕНИЕ: Colab runtime отличается от протестированного:')
    for item in runtime_warnings:
        print(' -', item)
else:
    print('Версии runtime совпадают с протестированной линией.')

# Быстрый реальный CUDA тест
x = torch.randn(1024, 1024, device='cuda', dtype=torch.float16)
y = x @ x
torch.cuda.synchronize()
print('CUDA вычисление: OK', tuple(y.shape))

# Проверяем, что RMVPE действительно загружается на GPU и обрабатывает аудио
os.environ['DDSP_ROOT'] = '/content/DDSP-SVC'
os.chdir('/content/DDSP-SVC')
from ddsp.vocoder import F0_Extractor

rmvpe = Path('pretrain/rmvpe/model.pt')
if not rmvpe.exists():
    raise FileNotFoundError(f'RMVPE не найден: {rmvpe}')

audio = (0.05 * np.sin(2 * np.pi * 220 * np.arange(44100, dtype=np.float32) / 44100)).astype(np.float32)
f0 = F0_Extractor('rmvpe', 44100, 512, 50.0, 1100.0).extract(audio, uv_interp=True, device='cuda')
print('RMVPE CUDA: OK, кадров:', len(f0))
print('GPU smoke test завершён успешно')


In [ ]:
import os
os.environ['DDSP_ROOT'] = '/content/DDSP-SVC'
%cd /content/DDSP-SVC
!python /content/DDSP-SVC-ReFlow-Colab/colab_app.py
